## Retrieval avance

1. **Similarity Search** : Recherche classique par similarite
2. **MMR** : Equilibre pertinence et diversite 
2. **Hybrid Search** : Combine BM25 (keywords) + recherche vectorielle 

In [1]:
# Installation
# !pip install langchain-text-splitters sentence-transformers chromadb rank-bm25

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

c:\Users\Administrateur\Documents\M2i_CDSD_TDTP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tech_articles = [
    # Articles Python (20)
    {
        "content": "Python est un langage de programmation interprete et oriente objet. Il est connu pour sa syntaxe claire et lisible.",
        "category": "python",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Les decorateurs Python permettent de modifier le comportement des fonctions et classes de maniere elegante.",
        "category": "python",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Les comprehensions de listes en Python offrent une syntaxe concise pour creer des listes basees sur des sequences existantes.",
        "category": "python",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "Le GIL (Global Interpreter Lock) de Python limite l'execution parallele de threads sur plusieurs coeurs CPU.",
        "category": "python",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Les generateurs Python utilisent yield pour produire des valeurs de maniere lazy, economisant la memoire.",
        "category": "python",
        "year": 2023,
        "author": "Bob",
    },
    {
        "content": "Le type hints en Python ameliore la lisibilite du code et permet la verification statique avec mypy.",
        "category": "python",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Les context managers en Python avec 'with' garantissent la bonne gestion des ressources comme les fichiers.",
        "category": "python",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Python 3.12 introduit des ameliorations de performance significatives grace a l'optimiseur adaptatif.",
        "category": "python",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Les dataclasses Python simplifient la creation de classes pour stocker des donnees.",
        "category": "python",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "asyncio en Python permet la programmation asynchrone pour des operations I/O non-bloquantes.",
        "category": "python",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Le pattern matching (match-case) introduit en Python 3.10 ameliore la lisibilite des conditions complexes.",
        "category": "python",
        "year": 2023,
        "author": "Bob",
    },
    {
        "content": "Les metaclasses Python permettent de personnaliser la creation de classes de maniere avancee.",
        "category": "python",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Pydantic offre une validation de donnees robuste avec des types Python natifs.",
        "category": "python",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "FastAPI exploite les type hints Python pour generer automatiquement de la documentation OpenAPI.",
        "category": "python",
        "year": 2023,
        "author": "Bob",
    },
    {
        "content": "Les f-strings Python offrent un formatage de chaines plus lisible et performant que format().",
        "category": "python",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Le module pathlib Python fournit une approche orientee objet pour manipuler les chemins de fichiers.",
        "category": "python",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Les enums Python permettent de definir des ensembles de constantes nommees de maniere elegante.",
        "category": "python",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le module functools Python offre des outils pour la programmation fonctionnelle comme partial et reduce.",
        "category": "python",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "Les annotations Python au runtime peuvent etre exploitees pour creer des frameworks dynamiques.",
        "category": "python",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Le protocole de buffer Python permet une manipulation efficace de donnees binaires sans copie.",
        "category": "python",
        "year": 2023,
        "author": "Bob",
    },
    # Articles Machine Learning (25)
    {
        "content": "Le machine learning est une branche de l'intelligence artificielle qui permet aux ordinateurs d'apprendre sans etre explicitement programmes.",
        "category": "ml",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Les reseaux de neurones profonds (deep learning) utilisent des couches multiples pour extraire des features hierarchiques.",
        "category": "ml",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Le gradient descent est un algorithme d'optimisation iteratif pour minimiser une fonction de cout.",
        "category": "ml",
        "year": 2023,
        "author": "Bob",
    },
    {
        "content": "Le overfitting survient quand un modele apprend trop les details du training set et generalise mal.",
        "category": "ml",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "La regularisation L1 et L2 aide a prevenir l'overfitting en penalisant les poids eleves du modele.",
        "category": "ml",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Le dropout est une technique de regularisation qui desactive aleatoirement des neurones pendant l'entrainement.",
        "category": "ml",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Les CNNs (Convolutional Neural Networks) excellent dans le traitement d'images grace a leurs couches convolutives.",
        "category": "ml",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Les RNNs (Recurrent Neural Networks) sont adaptes aux sequences grace a leur memoire interne.",
        "category": "ml",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Les Transformers utilisent le mecanisme d'attention pour traiter les sequences en parallele.",
        "category": "ml",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "BERT est un modele Transformer pre-entraine pour diverses taches de NLP par fine-tuning.",
        "category": "ml",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "GPT (Generative Pre-trained Transformer) est concu pour la generation de texte autoregressive.",
        "category": "ml",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Le transfer learning permet de reutiliser des modeles pre-entraines sur de nouvelles taches.",
        "category": "ml",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "L'augmentation de donnees artificiellement accroit la taille du dataset pour ameliorer la generalisation.",
        "category": "ml",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "Le batch normalization normalise les activations de chaque couche pour stabiliser l'entrainement.",
        "category": "ml",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Les GANs (Generative Adversarial Networks) utilisent deux reseaux competitifs pour generer des donnees realistes.",
        "category": "ml",
        "year": 2023,
        "author": "Bob",
    },
    {
        "content": "L'apprentissage par renforcement entraine des agents a maximiser des recompenses dans un environnement.",
        "category": "ml",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Le Q-learning est un algorithme classique d'apprentissage par renforcement sans modele.",
        "category": "ml",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Les autoencoders sont des reseaux de neurones pour l'apprentissage de representations compressees.",
        "category": "ml",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le fine-tuning adapte un modele pre-entraine a une tache specifique avec peu de donnees.",
        "category": "ml",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "La validation croisee k-fold evalue la performance du modele sur differents sous-ensembles des donnees.",
        "category": "ml",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "L'ensemble learning combine plusieurs modeles pour ameliorer la precision et la robustesse.",
        "category": "ml",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le bagging reduit la variance en entrainant plusieurs modeles sur des echantillons bootstrap.",
        "category": "ml",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "Le boosting reduit le biais en entrainant sequentiellement des modeles sur les erreurs des precedents.",
        "category": "ml",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "XGBoost est une implementation optimisee du gradient boosting tres utilisee en competitions.",
        "category": "ml",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Les metriques comme precision, recall et F1-score evaluent les performances des classifieurs.",
        "category": "ml",
        "year": 2023,
        "author": "Alice",
    },
    # Articles RAG (25)
    {
        "content": "Le RAG (Retrieval-Augmented Generation) combine recuperation d'informations et generation par LLM.",
        "category": "rag",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Les embeddings vectoriels representent le texte dans un espace semantique pour la recherche de similarite.",
        "category": "rag",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "ChromaDB est une base vectorielle open-source optimisee pour les applications RAG.",
        "category": "rag",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Le chunking decoupe les documents en morceaux optimaux pour la recherche vectorielle.",
        "category": "rag",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le RecursiveCharacterTextSplitter decoupe le texte en respectant sa structure hierarchique.",
        "category": "rag",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Le SemanticChunker utilise les embeddings pour detecter les ruptures semantiques lors du chunking.",
        "category": "rag",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "La similarite cosinus mesure l'angle entre deux vecteurs dans l'espace d'embeddings.",
        "category": "rag",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le MMR (Maximum Marginal Relevance) equilibre pertinence et diversite des resultats de recherche.",
        "category": "rag",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Le hybrid search combine BM25 (keywords) et recherche vectorielle (semantique) pour plus de robustesse.",
        "category": "rag",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "BM25 est un algorithme de ranking base sur TF-IDF optimise pour la recherche d'informations.",
        "category": "rag",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le reranking utilise des cross-encoders pour reordonner finement les documents recuperes.",
        "category": "rag",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Les bi-encoders encodent query et documents separement, permettant le pre-calcul des embeddings.",
        "category": "rag",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Les cross-encoders encodent query et document ensemble pour un scoring plus precis mais plus lent.",
        "category": "rag",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le Self-Query Retriever extrait automatiquement les filtres de metadonnees depuis la requete.",
        "category": "rag",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Le Multi-Query Retriever genere plusieurs variantes de la question pour enrichir les resultats.",
        "category": "rag",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Le Parent Document Retriever recherche via petits chunks mais retourne les documents parents complets.",
        "category": "rag",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "HyDE genere un document hypothetique repondant a la question pour ameliorer la recherche.",
        "category": "rag",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Le prompt engineering pour RAG structure le contexte et la question pour optimiser les reponses LLM.",
        "category": "rag",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "L'evaluation du RAG mesure la pertinence des documents recuperes et la qualite des reponses generees.",
        "category": "rag",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le chunk overlap preserve le contexte aux frontieres entre chunks consecutifs.",
        "category": "rag",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Les metadonnees enrichies facilitent le filtrage et le routing des requetes dans le RAG.",
        "category": "rag",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "FAISS (Facebook AI Similarity Search) est une bibliotheque optimisee pour la recherche vectorielle a grande echelle.",
        "category": "rag",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Pinecone est une base vectorielle cloud geree, optimisee pour les applications production.",
        "category": "rag",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Weaviate combine recherche vectorielle et graphe de connaissances pour des requetes complexes.",
        "category": "rag",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Le contexte window du LLM limite la quantite de chunks pouvant etre inclus dans le prompt.",
        "category": "rag",
        "year": 2024,
        "author": "Bob",
    },
    # Articles DevOps (20)
    {
        "content": "Docker permet d'empaqueter des applications avec leurs dependances dans des conteneurs portables.",
        "category": "devops",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Kubernetes orchestre des conteneurs a grande echelle avec auto-scaling et self-healing.",
        "category": "devops",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "CI/CD (Continuous Integration/Continuous Deployment) automatise le pipeline de developpement.",
        "category": "devops",
        "year": 2023,
        "author": "Bob",
    },
    {
        "content": "Terraform permet l'Infrastructure as Code pour gerer l'infrastructure cloud de maniere declarative.",
        "category": "devops",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Les microservices decoupent une application en services independants deployables separement.",
        "category": "devops",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Prometheus collecte et stocke des metriques time-series pour le monitoring d'infrastructure.",
        "category": "devops",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Grafana visualise les metriques et logs pour observer l'etat des systemes en temps reel.",
        "category": "devops",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Ansible automatise la configuration et le deploiement d'infrastructure de maniere agentless.",
        "category": "devops",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "GitOps utilise Git comme source de verite pour l'etat desire de l'infrastructure.",
        "category": "devops",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Les service mesh comme Istio gerent la communication entre microservices avec observabilite.",
        "category": "devops",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Les blue-green deployments minimisent les downtime en basculant entre deux environnements.",
        "category": "devops",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Les canary releases deployent progressivement les nouvelles versions a un sous-ensemble d'utilisateurs.",
        "category": "devops",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Les feature flags permettent d'activer/desactiver des fonctionnalites sans deploiement.",
        "category": "devops",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Le monitoring distribue avec ELK stack (Elasticsearch, Logstash, Kibana) agrege les logs.",
        "category": "devops",
        "year": 2023,
        "author": "Charlie",
    },
    {
        "content": "Les health checks verifient l'etat des services pour le load balancing et auto-healing.",
        "category": "devops",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le zero-downtime deployment permet de mettre a jour l'application sans interruption de service.",
        "category": "devops",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "Les secrets management tools comme Vault securisent les credentials et cles API.",
        "category": "devops",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Le chaos engineering teste la resilience en injectant deliberement des pannes.",
        "category": "devops",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Les SLOs (Service Level Objectives) definissent des cibles mesurables de fiabilite.",
        "category": "devops",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "Le serverless (Functions as a Service) execute du code sans gerer l'infrastructure.",
        "category": "devops",
        "year": 2024,
        "author": "Charlie",
    },
    # Articles Database (15)
    {
        "content": "PostgreSQL est une base de donnees relationnelle open-source avec support JSON et requetes complexes.",
        "category": "database",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "MongoDB est une base NoSQL orientee documents, stockant les donnees en format JSON/BSON.",
        "category": "database",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "Redis est un store cle-valeur en memoire ultra-rapide utilise pour le caching et les sessions.",
        "category": "database",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Les index B-tree accelerent les recherches dans les bases relationnelles en structurant les cles.",
        "category": "database",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Le ACID (Atomicity, Consistency, Isolation, Durability) garantit la fiabilite des transactions.",
        "category": "database",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "Le sharding partitionne horizontalement les donnees pour distribuer la charge sur plusieurs serveurs.",
        "category": "database",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "La replication master-slave copie les donnees du serveur maitre vers les replicas en lecture seule.",
        "category": "database",
        "year": 2024,
        "author": "Bob",
    },
    {
        "content": "Les transactions distribuees avec 2PC (Two-Phase Commit) coordonnent les operations multi-bases.",
        "category": "database",
        "year": 2023,
        "author": "Alice",
    },
    {
        "content": "Le CAP theorem enonce qu'un systeme distribue ne peut garantir simultanement Consistency, Availability et Partition tolerance.",
        "category": "database",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Les CQL (Cassandra Query Language) permet de requeter Cassandra avec une syntaxe proche de SQL.",
        "category": "database",
        "year": 2023,
        "author": "Bob",
    },
    {
        "content": "Le denormalization en NoSQL duplique les donnees pour optimiser les lectures au detriment de l'espace.",
        "category": "database",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "Les OLTP (Online Transaction Processing) optimisent les operations transactionnelles frequentes.",
        "category": "database",
        "year": 2024,
        "author": "Charlie",
    },
    {
        "content": "Les OLAP (Online Analytical Processing) optimisent les requetes analytiques sur gros volumes.",
        "category": "database",
        "year": 2023,
        "author": "Bob",
    },
    {
        "content": "Les column-stores comme Cassandra stockent les donnees par colonnes pour des lectures analytiques rapides.",
        "category": "database",
        "year": 2024,
        "author": "Alice",
    },
    {
        "content": "L'eventual consistency en NoSQL garantit que les replicas convergent vers le meme etat finalement.",
        "category": "database",
        "year": 2023,
        "author": "Charlie",
    },
]

# Convertir en Documents LangChain
documents = [
    Document(
        page_content=article["content"],
        metadata={
            "category": article["category"],
            "year": article["year"],
            "author": article["author"],
        },
    )
    for article in tech_articles
]

print(f"Dataset : {len(documents)} documents")
print(f"\nCategories : {set(d.metadata['category'] for d in documents)}")
print(f"Auteurs : {set(d.metadata['author'] for d in documents)}")
print(f"Annees : {set(d.metadata['year'] for d in documents)}")

Dataset : 105 documents

Categories : {'ml', 'database', 'devops', 'python', 'rag'}
Auteurs : {'Charlie', 'Alice', 'Bob'}
Annees : {2024, 2023}


## Préparation embeddings et vector store

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=documents, embedding=embeddings, collection_name="tech_articles"
)

C:\Users\Administrateur\AppData\Local\Temp\ipykernel_9236\1711034400.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10270.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Technique 1 : Similarity Search

In [5]:
query = "Comment optimiser les performances de python ?"

results = vectorstore.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results, 1):
    print(f"{i} | Score : {score}")
    print(f"content : {doc.page_content[:100]}")

1 | Score : 0.3988056480884552
content : Python 3.12 introduit des ameliorations de performance significatives grace a l'optimiseur adaptatif
2 | Score : 0.7256773710250854
content : Le module functools Python offre des outils pour la programmation fonctionnelle comme partial et red
3 | Score : 0.7924287915229797
content : Les annotations Python au runtime peuvent etre exploitees pour creer des frameworks dynamiques.
4 | Score : 0.883407711982727
content : Le GIL (Global Interpreter Lock) de Python limite l'execution parallele de threads sur plusieurs coe
5 | Score : 0.8840458393096924
content : Les context managers en Python avec 'with' garantissent la bonne gestion des ressources comme les fi


## Technique 2 : MMR

In [9]:
lambda_values = [0.3, 0.5, 0.7, 1.0]

for lambda_val in lambda_values:
    print(f"MMR : {lambda_val}")
    # lambda = 0 : max diversite, lambda=1 : max pertinence

    mmr_results = vectorstore.max_marginal_relevance_search(
        query, k=5, fetch_k=20, lambda_mult=lambda_val
    )

    for i, doc in enumerate(mmr_results, 1):
        print(f"{i} | content : {doc.page_content[:100]}")

MMR : 0.3
1 | content : Python 3.12 introduit des ameliorations de performance significatives grace a l'optimiseur adaptatif
2 | content : Le type hints en Python ameliore la lisibilite du code et permet la verification statique avec mypy.
3 | content : asyncio en Python permet la programmation asynchrone pour des operations I/O non-bloquantes.
4 | content : Les f-strings Python offrent un formatage de chaines plus lisible et performant que format().
5 | content : ChromaDB est une base vectorielle open-source optimisee pour les applications RAG.
MMR : 0.5
1 | content : Python 3.12 introduit des ameliorations de performance significatives grace a l'optimiseur adaptatif
2 | content : Le module functools Python offre des outils pour la programmation fonctionnelle comme partial et red
3 | content : Le GIL (Global Interpreter Lock) de Python limite l'execution parallele de threads sur plusieurs coe
4 | content : Les f-strings Python offrent un formatage de chaines plus lisible et performant

## Technique 3 : Hybrid search

- combine recherche par keywords et recherche semantique

In [7]:
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], weights=[0.8, 0.2]
)

In [11]:
query_ensemble = "GIL Python Interpreter Lock"

hybrid_results = ensemble_retriever.invoke(query)

for i, doc in enumerate(hybrid_results, 1):
    print(f"{i:<2} | content : {doc.page_content[:100]}")

1  | content : Les metriques comme precision, recall et F1-score evaluent les performances des classifieurs.
2  | content : Le denormalization en NoSQL duplique les donnees pour optimiser les lectures au detriment de l'espac
3  | content : Le prompt engineering pour RAG structure le contexte et la question pour optimiser les reponses LLM.
4  | content : Les index B-tree accelerent les recherches dans les bases relationnelles en structurant les cles.
5  | content : Le batch normalization normalise les activations de chaque couche pour stabiliser l'entrainement.
6  | content : Python 3.12 introduit des ameliorations de performance significatives grace a l'optimiseur adaptatif
7  | content : Le module functools Python offre des outils pour la programmation fonctionnelle comme partial et red
8  | content : Les annotations Python au runtime peuvent etre exploitees pour creer des frameworks dynamiques.
9  | content : Le GIL (Global Interpreter Lock) de Python limite l'execution parallele de 